In [ ]:
%pip install yfinance pandas numpy scipy scikit-learn textblob vaderSentiment python-dotenv tqdm joblib
%pip install matplotlib plotly
# Optional (for LLM/agent + news):
%pip install openai langchain langchain-community langchain-openai
%pip install newsapi-python


In [1]:

import os
import math
import json
import time
import warnings
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import yfinance as yf

from tqdm import tqdm
from joblib import Memory

from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize

# Sentiment (optional)
try:
    from textblob import TextBlob
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _HAS_SENT = True
except Exception:
    _HAS_SENT = False

# News API (optional)
try:
    from newsapi import NewsApiClient
    _HAS_NEWS = True
except Exception:
    _HAS_NEWS = False

# .env config
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("RebalanceAI")

# Global config / knobs
TIMEZONE = "America/New_York"
TRADING_DAYS = 252

# Risk-free can be set via ENV or fallback default
RISK_FREE = float(os.getenv("RISK_FREE_RATE", "0.04"))  # 4% default; change via .env if needed

# Optional API keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", None)
NEWS_API_KEY = os.getenv("NEWS_API_KEY", None)

# Disk cache folder (feel free to change or disable by setting to None)
CACHE_DIR = os.getenv("REBALANCEAI_CACHE_DIR", "/mnt/data/rebalance_cache")
memory = Memory(location=CACHE_DIR, verbose=0) if CACHE_DIR else None

# Risk profiles (tunable)
RISK_PROFILES = {
    "conservative": {
        "max_single_stock": 0.10,
        "max_sector": 0.25,
        "min_stocks": 5,
        "beta_preference": "low",
        "sentiment_multiplier": 0.8,
        "risk_aversion": 0.7,
        "target_sharpe": 1.0,
    },
    "moderate": {
        "max_single_stock": 0.15,
        "max_sector": 0.35,
        "min_stocks": 5,
        "beta_preference": "neutral",
        "sentiment_multiplier": 1.0,
        "risk_aversion": 0.5,
        "target_sharpe": 1.3,
    },
    "aggressive": {
        "max_single_stock": 0.25,
        "max_sector": 0.5,
        "min_stocks": 4,
        "beta_preference": "high",
        "sentiment_multiplier": 1.2,
        "risk_aversion": 0.3,
        "target_sharpe": 1.6,
    },
}

# Alpha scoring weights centralized here
ALPHA_WEIGHTS = {
    "fundamentals": 0.4,
    "technicals": 0.4,
    "sentiment":   0.2,
}

TECH_WEIGHTS = {  # sum to ~1 inside technical sub-score
    "sharpe": 0.45,
    "rsi":    0.15,
    "drawdown": 0.20,
    "momentum_3m": 0.20,
}

FUND_WEIGHTS = {
    "roe": 0.25,
    "roa": 0.10,
    "de_ratio": 0.10,     # lower is better
    "current_ratio": 0.10,
    "rev_growth": 0.20,
    "earnings_growth": 0.15,
    "pe": 0.10,          # lower is better
}

SENT_WEIGHTS = {
    "avg_sentiment": 0.45,
    "sent_momentum": 0.25,
    "toxicity": 0.10,   # lower is better
    "attention": 0.20,
}

# Helper for caching decorator
def cacheable(func):
    if memory:
        return memory.cache(func)
    return func


In [3]:

def safe_div(a, b, default=np.nan):
    try:
        return a / b if (b is not None and b != 0) else default
    except Exception:
        return default

def normalize_metric(value, direction="higher", low=None, high=None, clip=True):
    """Map value to 0..100 given [low, high] and whether higher is better.
    If bounds are missing, falls back to heuristic scaling.
    """
    if value is None or (isinstance(value, float) and (np.isnan(value) or np.isinf(value))):
        return 50.0
    if low is None or high is None or low == high:
        # Heuristic: logistic-like squashing around 0
        x = float(value)
        score = 100 / (1 + math.exp(-x))
        return float(np.clip(score, 0, 100))
    # linear scale
    x = (value - low) / (high - low)
    x = np.clip(x, 0, 1) if clip else x
    score = 100 * x if direction == "higher" else 100 * (1 - x)
    return float(np.clip(score, 0, 100))

def annualize_vol(daily_ret: pd.Series) -> float:
    return float(np.nanstd(daily_ret, ddof=1) * math.sqrt(TRADING_DAYS))

def sharpe_ratio(daily_ret: pd.Series, risk_free=RISK_FREE) -> float:
    mu = float(np.nanmean(daily_ret)) * TRADING_DAYS
    vol = annualize_vol(daily_ret)
    if vol == 0 or np.isnan(vol):
        return np.nan
    return (mu - risk_free) / vol

def rsi(series: pd.Series, period=14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -1 * delta.clip(upper=0)
    ma_up = up.ewm(com=period-1, adjust=False).mean()
    ma_down = down.ewm(com=period-1, adjust=False).mean()
    rs = ma_up / ma_down
    return 100 - (100 / (1 + rs))

def rolling_max_drawdown(series: pd.Series) -> float:
    cummax = series.cummax()
    dd = (series - cummax) / cummax
    return float(dd.min())  # negative number


In [2]:

@cacheable
def fetch_history(ticker: str, period="5y", interval="1d") -> pd.DataFrame:
    df = yf.Ticker(ticker).history(period=period, interval=interval, auto_adjust=True)
    if not isinstance(df, pd.DataFrame) or df.empty:
        raise ValueError(f"No price data for {ticker}")
    return df

@cacheable
def fetch_fundamentals(ticker: str) -> Dict[str, pd.DataFrame]:
    t = yf.Ticker(ticker)
    # yfinance returns wide dataframes with columns as periods
    inc = getattr(t, "quarterly_financials", None)
    if inc is None or (isinstance(inc, pd.DataFrame) and inc.empty):
        inc = getattr(t, "quarterly_income_stmt", pd.DataFrame())
    bs  = getattr(t, "quarterly_balance_sheet", pd.DataFrame())
    cf  = getattr(t, "quarterly_cashflow", pd.DataFrame())
    out = {
        "income": inc if isinstance(inc, pd.DataFrame) else pd.DataFrame(),
        "balance": bs if isinstance(bs, pd.DataFrame) else pd.DataFrame(),
        "cashflow": cf if isinstance(cf, pd.DataFrame) else pd.DataFrame(),
        "info": t.info if hasattr(t, "info") else {},
    }
    return out

def ttm_sum(df: pd.DataFrame, row_name: str, as_of_date: pd.Timestamp=None, lag_days:int=45):
    """
    Sum the last 4 reported quarters **available as of** (report_date + lag_days) <= as_of_date.
    Handles tz-aware vs tz-naive comparisons by normalizing to tz-naive.
    """
    if df is None or df.empty or row_name not in df.index:
        return None
    series = df.loc[row_name]
    series = series.sort_index(ascending=False)

    if as_of_date is None:
        vals = series.dropna().values[:4]
        return float(np.nansum(vals)) if len(vals) else None

    # normalize as_of_date to tz-naive
    as_of_naive = pd.Timestamp(as_of_date)
    if getattr(as_of_naive, "tzinfo", None) is not None:
        try:
            as_of_naive = as_of_naive.tz_convert(None)
        except Exception:
            as_of_naive = as_of_naive.tz_localize(None)

    vals = []
    for col in series.index:
        cutoff = pd.Timestamp(col) + pd.Timedelta(days=lag_days)  # tz-naive by construction
        if cutoff <= as_of_naive:
            v = series.loc[col]
            if not (pd.isna(v) or v is None):
                vals.append(float(v))
            if len(vals) == 4:
                break
    return float(np.nansum(vals)) if len(vals) == 4 else None

def get_sentiment_from_newsapi(query: str, from_date: str, to_date: str, page_size: int=50) -> Dict[str, float]:
    if not (_HAS_NEWS and NEWS_API_KEY):
        return {"avg": 0.0, "momentum": 0.0, "toxicity": 0.0, "attention": 0.0}
    client = NewsApiClient(api_key=NEWS_API_KEY)
    try:
        articles = client.get_everything(
            q=query, language="en", from_param=from_date, to=to_date,
            sort_by="relevancy", page_size=min(page_size, 100)
        )
    except Exception:
        return {"avg": 0.0, "momentum": 0.0, "toxicity": 0.0, "attention": 0.0}

    hits = articles.get("articles", []) if isinstance(articles, dict) else []
    if not hits:
        return {"avg": 0.0, "momentum": 0.0, "toxicity": 0.0, "attention": 0.0}
    scores, tox = [], 0
    sid = SentimentIntensityAnalyzer() if _HAS_SENT else None
    mid_point = pd.to_datetime(from_date) + (pd.to_datetime(to_date) - pd.to_datetime(from_date))/2
    early_scores, late_scores = [], []
    for a in hits:
        txt = " ".join([a.get("title",""), a.get("description","")]).strip()
        if not txt:
            continue
        s = 0.0
        if sid:
            s = sid.polarity_scores(txt)["compound"]
        else:
            s = TextBlob(txt).sentiment.polarity if _HAS_SENT else 0.0
        scores.append(s)
        lower = txt.lower()
        tox += sum(1 for k in ["fraud","scandal","probe","lawsuit","ban","recall","downgrade"] if k in lower)
        t = pd.to_datetime(a.get("publishedAt")) if a.get("publishedAt") else None
        (early_scores if (t and t < mid_point) else late_scores).append(s)
    avg = float(np.mean(scores)) if scores else 0.0
    momentum = float(np.mean(late_scores) - np.mean(early_scores)) if (late_scores and early_scores) else 0.0
    return {"avg": avg, "momentum": momentum, "toxicity": float(tox), "attention": float(len(scores))}


In [11]:

@cacheable
def fetch_info_sector(ticker: str) -> Optional[str]:
    try:
        info = yf.Ticker(ticker).info
        return info.get("sector")
    except Exception:
        return None


In [10]:

def analyze_stock(ticker: str, days_lookback: int = 30, as_of: Optional[pd.Timestamp]=None, risk_profile: str="moderate") -> Dict:
    as_of = as_of or pd.Timestamp.utcnow()
    try:
        prices = fetch_history(ticker, period="2y", interval="1d")
    except Exception as e:
        logger.warning(f"{ticker}: price fetch failed: {e}")
        return {"ticker": ticker, "error": str(e)}

    close = prices["Close"].dropna()
    if close.empty:
        return {"ticker": ticker, "error": "no close prices"}

    # Technicals
    daily_ret = close.pct_change().dropna()
    ann_vol = annualize_vol(daily_ret)
    sr = sharpe_ratio(daily_ret, risk_free=RISK_FREE)
    rsi14 = rsi(close, 14).iloc[-1] if len(close) >= 15 else np.nan
    dd = rolling_max_drawdown(close)  # negative number
    if len(close) > 65:
        momentum_3m = float(close.iloc[-1] / close.iloc[-65] - 1)
    else:
        momentum_3m = np.nan

    # Fundamentals
    f = fetch_fundamentals(ticker)
    inc, bs, cf, info = f.get("income"), f.get("balance"), f.get("cashflow"), f.get("info", {})
    rev_ttm = ttm_sum(inc, "Total Revenue", as_of_date=as_of)
    net_ttm = ttm_sum(inc, "Net Income", as_of_date=as_of)
    assets = bs.loc["Total Assets"].dropna().sort_index().iloc[-1] if (isinstance(bs, pd.DataFrame) and "Total Assets" in bs.index and not bs.loc["Total Assets"].dropna().empty) else np.nan
    equity = bs.loc["Total Stockholder Equity"].dropna().sort_index().iloc[-1] if (isinstance(bs, pd.DataFrame) and "Total Stockholder Equity" in bs.index and not bs.loc["Total Stockholder Equity"].dropna().empty) else np.nan
    current_assets = bs.loc["Total Current Assets"].dropna().sort_index().iloc[-1] if (isinstance(bs, pd.DataFrame) and "Total Current Assets" in bs.index and not bs.loc["Total Current Assets"].dropna().empty) else np.nan
    current_liab = bs.loc["Total Current Liabilities"].dropna().sort_index().iloc[-1] if (isinstance(bs, pd.DataFrame) and "Total Current Liabilities" in bs.index and not bs.loc["Total Current Liabilities"].dropna().empty) else np.nan
    total_debt = bs.loc["Total Debt"].dropna().sort_index().iloc[-1] if (isinstance(bs, pd.DataFrame) and "Total Debt" in bs.index and not bs.loc["Total Debt"].dropna().empty) else np.nan

    roe = safe_div(net_ttm, equity)
    roa = safe_div(net_ttm, assets)
    de_ratio = safe_div(total_debt, equity)
    curr_ratio = safe_div(current_assets, current_liab)

    def prior_ttm(df, row):
        if df is None or df.empty or row not in df.index:
            return None
        series = df.loc[row].sort_index(ascending=False).dropna()
        if len(series) < 8:
            return None
        return float(np.nansum(series.values[4:8]))
    rev_prior = prior_ttm(inc, "Total Revenue")
    net_prior = prior_ttm(inc, "Net Income")
    rev_growth = safe_div((rev_ttm - rev_prior), rev_prior) if (rev_ttm is not None and rev_prior not in (None,0)) else np.nan
    earn_growth = safe_div((net_ttm - net_prior), net_prior) if (net_ttm is not None and net_prior not in (None,0)) else np.nan

    pe = info.get("trailingPE") or info.get("forwardPE") or np.nan

    end_date = pd.Timestamp(as_of).normalize()
    start_date = end_date - pd.Timedelta(days=days_lookback)
    sent = get_sentiment_from_newsapi(f"{ticker} stock", start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d"))

    tech_score = (
        TECH_WEIGHTS["sharpe"] * normalize_metric(sr, "higher", low=-1.0, high=3.0) +
        TECH_WEIGHTS["rsi"]    * normalize_metric(50-abs(50 - (rsi14 if not np.isnan(rsi14) else 50)), "higher", low=0, high=50) +
        TECH_WEIGHTS["drawdown"] * normalize_metric(dd, "higher", low=-0.7, high=0.0) +
        TECH_WEIGHTS["momentum_3m"] * normalize_metric(momentum_3m if not np.isnan(momentum_3m) else 0, "higher", low=-0.3, high=0.6)
    )

    fund_score = (
        FUND_WEIGHTS["roe"] * normalize_metric(roe if roe is not None else np.nan, "higher", low=-0.2, high=0.4) +
        FUND_WEIGHTS["roa"] * normalize_metric(roa if roa is not None else np.nan, "higher", low=-0.1, high=0.2) +
        FUND_WEIGHTS["de_ratio"] * normalize_metric(de_ratio if de_ratio is not None else np.nan, "lower", low=0.0, high=3.0) +
        FUND_WEIGHTS["current_ratio"] * normalize_metric(curr_ratio if curr_ratio is not None else np.nan, "higher", low=0.5, high=3.0) +
        FUND_WEIGHTS["rev_growth"] * normalize_metric(rev_growth if not np.isnan(rev_growth) else 0, "higher", low=-0.3, high=0.5) +
        FUND_WEIGHTS["earnings_growth"] * normalize_metric(earn_growth if not np.isnan(earn_growth) else 0, "higher", low=-0.3, high=0.5) +
        FUND_WEIGHTS["pe"] * normalize_metric(pe if not np.isnan(pe) else 25, "lower", low=5, high=60)
    )

    prof = RISK_PROFILES.get(risk_profile, RISK_PROFILES["moderate"])
    sent_score_raw = (
        SENT_WEIGHTS["avg_sentiment"] * normalize_metric(sent["avg"], "higher", low=-0.5, high=0.5) +
        SENT_WEIGHTS["sent_momentum"] * normalize_metric(sent["momentum"], "higher", low=-0.2, high=0.2) +
        SENT_WEIGHTS["toxicity"] * normalize_metric(sent["toxicity"], "lower", low=0, high=15) +
        SENT_WEIGHTS["attention"] * normalize_metric(sent["attention"], "higher", low=0, high=60)
    )
    sent_score = float(np.clip(sent_score_raw * prof["sentiment_multiplier"], 0, 100))

    alpha = (
        ALPHA_WEIGHTS["fundamentals"] * fund_score +
        ALPHA_WEIGHTS["technicals"] * tech_score +
        ALPHA_WEIGHTS["sentiment"]   * sent_score
    )

    flags = []
    if ann_vol > 0.45: flags.append("High volatility")
    if dd < -0.4: flags.append("Deep drawdown")
    if de_ratio and de_ratio > 2.5: flags.append("Elevated leverage")
    if (not np.isnan(rsi14)) and (rsi14 > 70): flags.append("Overbought (RSI)")
    if (not np.isnan(rsi14)) and (rsi14 < 30): flags.append("Oversold (RSI)")

    return {
        "ticker": ticker,
        "as_of": as_of.isoformat(),
        "price": float(close.iloc[-1]),
        "daily_returns": daily_ret.tail(TRADING_DAYS),
        "ann_vol": ann_vol,
        "sharpe": sr,
        "rsi14": float(rsi14) if not np.isnan(rsi14) else None,
        "drawdown": dd,
        "momentum_3m": momentum_3m,
        "roe": roe, "roa": roa, "de_ratio": de_ratio, "current_ratio": curr_ratio,
        "rev_ttm": rev_ttm, "net_ttm": net_ttm,
        "rev_growth": rev_growth, "earnings_growth": earn_growth, "pe": pe,
        "sentiment": sent,
        "tech_score": tech_score, "fund_score": fund_score, "sent_score": sent_score,
        "alpha_score": float(alpha),
        "risk_flags": flags,
        "sector": (fetch_info_sector(ticker) or "Unknown"),
    }


In [9]:

def validate_ticker(t: str) -> str:
    t = t.strip().upper().replace("/", "-")
    mapping = {"FB": "META", "BRK.B": "BRK-B", "GOOG": "GOOGL"}
    return mapping.get(t, t)

def batch_analyze(tickers: List[str], risk_profile="moderate", as_of: Optional[pd.Timestamp]=None) -> pd.DataFrame:
    rows = []
    for t in tqdm(tickers, desc="Analyzing"):
        t0 = validate_ticker(t)
        res = analyze_stock(t0, risk_profile=risk_profile, as_of=as_of)
        if "error" in res:
            logger.warning(f"{t0} skipped: {res['error']}")
            continue
        rows.append(res)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).sort_values("alpha_score", ascending=False).reset_index(drop=True)
    return df

DEFAULT_UNIVERSE = [
    "AAPL","MSFT","NVDA","AMZN","META","GOOGL","AVGO","TSLA","LLY","JPM",
    "V","MA","COST","XOM","UNH","PG","HD","ORCL","BAC","KO","MRK","PEP",
    "ABBV","WMT","CVX","ADBE","NFLX","CRM","ACN","LIN","TXN","AMD","INTC",
    "DIS","NKE","PYPL","AMAT","QCOM","TMO","UPS","CAT","BA","GE","IBM","PFE"
]


In [8]:

def _prepare_cov_matrix(returns_df: pd.DataFrame) -> np.ndarray:
    lw = LedoitWolf().fit(returns_df.dropna())
    return lw.covariance_

def optimize_portfolio(df: pd.DataFrame, capital: float=10000, risk_profile: str="moderate") -> Dict:
    if df.empty:
        raise ValueError("Empty input DataFrame")

    prof = RISK_PROFILES.get(risk_profile, RISK_PROFILES["moderate"])
    max_single = prof["max_single_stock"]
    max_sector = prof["max_sector"]
    min_stocks = prof["min_stocks"]

    # Expected returns proxy: scaled alpha (0..100 -> 0..15%)
    exp_rets = (df["alpha_score"].values / 100.0) * 0.15
    tickers = df["ticker"].tolist()
    sectors = df["sector"].fillna("Unknown").tolist()

    # Build returns matrix
    rets = []
    for _, row in df.iterrows():
        s = row["daily_returns"]
        rets.append(s.values)
    returns_mat = pd.DataFrame(rets).T  # time x names
    returns_mat.columns = tickers
    returns_mat = returns_mat.dropna(how="all")
    cov = _prepare_cov_matrix(returns_mat)

    def port_vol(w):
        return float(np.sqrt(np.dot(w.T, np.dot(cov, w)) * TRADING_DAYS))

    def port_ret(w):
        return float(np.dot(w, exp_rets))

    def neg_sharpe(w):
        vol = port_vol(w)
        if vol == 0 or np.isnan(vol):
            return 1e6
        return - (port_ret(w) - RISK_FREE) / vol

    n = len(tickers)
    w0 = np.array([1.0/n] * n)

    cons = [{"type":"eq", "fun": lambda w: np.sum(w) - 1.0}]

    unique_sectors = sorted(set(sectors))
    for sec in unique_sectors:
        idx = [i for i,s in enumerate(sectors) if s == sec]
        if not idx:
            continue
        cons.append({
            "type": "ineq",
            "fun": (lambda idx=idx: lambda w: max_sector - np.sum(w[idx]))()
        })

    bounds = [(0.0, max_single)] * n

    res = minimize(neg_sharpe, w0, method="SLSQP", bounds=bounds, constraints=cons, options={"maxiter": 1000, "ftol":1e-9})
    if (not res.success) or np.any(np.isnan(res.x)):
        logger.warning(f"Optimizer failed: {res.message}")
        w = np.array([min(max_single, 1.0/n)]*n)
        w = w / w.sum()
    else:
        w = res.x

    active = (w > 1e-4).sum()
    if active < min_stocks:
        order = np.argsort(-df["alpha_score"].values)
        w = np.zeros_like(w)
        w[order[:min_stocks]] = 1.0 / min_stocks

    alloc = (w * capital)
    expected_ret = port_ret(w)
    expected_vol = port_vol(w)
    expected_sharpe = (expected_ret - RISK_FREE) / expected_vol if expected_vol else np.nan

    sector_alloc = {}
    for wi, sec in zip(w, sectors):
        sector_alloc[sec] = sector_alloc.get(sec, 0.0) + wi

    out = {
        "tickers": tickers,
        "weights": w.tolist(),
        "allocations": {t: float(a) for t,a in zip(tickers, alloc)},
        "expected_return": expected_ret,
        "expected_vol": expected_vol,
        "expected_sharpe": expected_sharpe,
        "sector_weights": sector_alloc,
        "capital": float(capital),
        "risk_profile": risk_profile,
    }
    return out


In [7]:

def monte_carlo(port: Dict, horizon_days: int=TRADING_DAYS, num_sims: int=10000, market_baseline=0.10, shock_prob=0.005, shock_range=(-0.05, -0.02)):
    tickers = port["tickers"]
    weights = np.array(port["weights"])
    mat = []
    for t in tickers:
        try:
            s = fetch_history(t, period="2y", interval="1d")["Close"].pct_change().dropna().tail(TRADING_DAYS)
            mat.append(s.values)
        except Exception:
            mat.append(np.zeros(TRADING_DAYS))
    R = pd.DataFrame(mat).T
    R.columns = tickers
    R = R.fillna(0)

    mu_daily = R.mean().values
    mu_blend = 0.6 * mu_daily + 0.4 * (market_baseline / TRADING_DAYS)
    cov = _prepare_cov_matrix(R)

    try:
        chol = np.linalg.cholesky(cov)
    except np.linalg.LinAlgError:
        cov = cov + 1e-6 * np.eye(cov.shape[0])
        chol = np.linalg.cholesky(cov)

    start_val = port.get("capital", 10000.0)
    finals = np.zeros(num_sims)
    for k in range(num_sims):
        z = np.random.normal(size=(horizon_days, len(tickers)))
        correlated = z @ chol.T
        daily = mu_blend + correlated
        mask = (np.random.rand(horizon_days) < shock_prob)
        shock_vals = np.random.uniform(low=shock_range[0], high=shock_range[1], size=mask.sum())
        if mask.any():
            daily[mask, :] += shock_vals[:, None]
        port_daily = daily @ weights
        finals[k] = start_val * float(np.prod(1 + port_daily))

    pct = np.percentile(finals, [5, 25, 50, 75, 95])
    res = {
        "start": float(start_val),
        "mean_final": float(finals.mean()),
        "p5": float(pct[0]), "p25": float(pct[1]), "p50": float(pct[2]), "p75": float(pct[3]), "p95": float(pct[4]),
        "prob_loss": float((finals < start_val).mean()),
        "prob_10p": float((finals >= 1.10*start_val).mean()),
        "prob_20p": float((finals >= 1.20*start_val).mean()),
        "prob_50p": float((finals >= 1.50*start_val).mean()),
    }
    return res


In [6]:

def backtest_portfolio(start_date: str, capital: float=10000, risk_profile="moderate", hold_days: int=30, universe: Optional[List[str]]=None):
    start = pd.to_datetime(start_date).tz_localize("UTC")
    end = start + pd.Timedelta(days=hold_days)
    as_of = start  # for look-ahead control of fundamentals

    universe = universe or DEFAULT_UNIVERSE
    df = batch_analyze(universe, risk_profile=risk_profile, as_of=as_of)

    if df.empty or len(df) < 5:
        raise ValueError("Universe too small after analysis.")

    topN = max(RISK_PROFILES[risk_profile]["min_stocks"] * 2, 10)
    df_top = df.head(min(topN, len(df)))

    port = optimize_portfolio(df_top, capital=capital, risk_profile=risk_profile)

    port_val = capital
    weights = np.array(port["weights"])
    names = port["tickers"]

    gross = []
    for t in names:
        hist = fetch_history(t, period="5y", interval="1d")["Close"]
        hist = hist.tz_localize(None)
        try:
            s_px = hist.loc[:start.tz_convert(None)].iloc[-1]
            e_px = hist.loc[:end.tz_convert(None)].iloc[-1]
            gross.append(float(e_px / s_px - 1.0))
        except Exception:
            gross.append(0.0)
    gross = np.array(gross)
    realized = float(np.dot(weights, gross))
    final_val = capital * (1 + realized)

    try:
        spx = fetch_history("^GSPC", period="5y", interval="1d")["Close"].tz_localize(None)
        spx_s = spx.loc[:start.tz_convert(None)].iloc[-1]
        spx_e = spx.loc[:end.tz_convert(None)].iloc[-1]
        spx_ret = float(spx_e / spx_s - 1.0)
    except Exception:
        spx_ret = 0.0

    return {
        "start": start_date,
        "end": end.strftime("%Y-%m-%d"),
        "risk_profile": risk_profile,
        "capital": capital,
        "final_value": final_val,
        "portfolio_return": realized,
        "sp500_return": spx_ret,
        "alpha": realized - spx_ret,
        "beat_market": realized > spx_ret,
        "portfolio": port,
        "selected_df": df_top[["ticker","alpha_score","sector","sharpe","ann_vol","fund_score","tech_score","sent_score"]].reset_index(drop=True),
    }


In [12]:

# Example: analyze a single stock (as-of today)
res = analyze_stock("TSLA", risk_profile="moderate")
{ k: res[k] for k in ["ticker","price","alpha_score","tech_score","fund_score","sent_score","risk_flags"] }


{'ticker': 'TSLA',
 'price': 439.30999755859375,
 'alpha_score': 45.61985264597958,
 'tech_score': 51.01961451621184,
 'fund_score': 40.530017098737105,
 'sent_score': 45.0,
 'risk_flags': ['High volatility', 'Deep drawdown']}

In [ ]:

# Example: batch analyze a small universe
universe = ["AAPL","MSFT","NVDA","AMZN","META","GOOGL","AVGO","TSLA","LLY","JPM"]
df = batch_analyze(universe, risk_profile="moderate")
df.head(5)[["ticker","alpha_score","sector","sharpe","ann_vol","fund_score","tech_score","sent_score"]]


In [ ]:

# Example: optimize a portfolio from the batch
port = optimize_portfolio(df.head(10), capital=20000, risk_profile="moderate")
port


In [ ]:

# Example: Monte Carlo forecast for the optimized portfolio
mc = monte_carlo(port, horizon_days=252, num_sims=2000)
mc


In [ ]:

# Example: 30-day backtest starting from a past date
bt = backtest_portfolio("2024-09-18", capital=10000, risk_profile="moderate", hold_days=30, universe=universe)
{
    "portfolio_return": bt["portfolio_return"],
    "sp500_return": bt["sp500_return"],
    "alpha": bt["alpha"],
    "beat_market": bt["beat_market"]
}
